
#### Notebook : 03_coordinator_agent


##### 1. Purpose

The Coordinator Agent is responsible for analyzing a customer request and creating a structured execution plan for the multi-agent workflow.

The Coordinator Agent:

- Identifies the request type.
- Selects the required specialist agents.
- Defines the order in which agents should run.
- Defines dependencies between agent tasks.
- Validates the execution plan.
- Stores the validated CoordinatorResult in shared state.

The Coordinator Agent does not:

- Execute SQL queries.
- Call the prediction model.
- Perform vector search.
- Recommend retention actions.
- Generate the final customer-facing response.

Its responsibility is planning, not execution.


##### 2. Technologies Used

- Python
- Prompt engineering
- Pydantic validation
- JSON parsing
- Dependency injection
- Shared workflow state
- Databricks notebooks

##### 3. Input

The Coordinator Agent receives:

``` text

MultiAgentState
    └── user_request
```

It also receives an LLM invocation function that accepts a prompt and returns text.


##### 4. Output

The Coordinator Agent produces:

``` text

CoordinatorResult
    ├── agent_name
    ├── status
    ├── message
    ├── request_type
    └── execution_plan
            ├── task_id
            ├── agent_name
            ├── task_description
            └── depends_on

```


##### 5. Multi-Agent Architecture

```text


Customer Request
       │
       ▼
Coordinator Agent
       │
       ├── Analyze request
       ├── Identify request type
       ├── Select specialist agents
       ├── Define execution order
       └── Define dependencies
       │
       ▼
Validated CoordinatorResult
       │
       ▼
Shared Workflow State
       │
       ▼
Multi-Agent Orchestrator


```


##### 6. Load Dependencies

In [0]:
%run ./01_shared_schemas
%run ./02_shared_state_and_helpers


##### 7. Additional Imports

In [0]:
import json

from json import JSONDecodeError
from typing import Any, Callable, Dict, List, get_args
from pydantic import ValidationError


##### 8. LLM Function Type



In [0]:
LLMInvokeFunction = Callable[[str], str]


##### 9. Coordinator System Instructions

In [0]:
COORDINATOR_SYSTEM_INSTRUCTIONS = """
You are the Coordinator Agent in a telecom customer-support
multi-agent system.

Your only responsibility is to create a structured execution plan.

You must not:

- Answer the customer directly.
- Execute SQL queries.
- Call prediction models.
- Perform vector search.
- Recommend retention actions.
- Generate the final customer-facing response.

You must determine:

1. The request type.
2. The specialist agents required.
3. The correct execution order.
4. The dependencies between tasks.

Available specialist agents:

- sql_agent
- prediction_agent
- vector_search_agent
- retention_agent
- final_response_agent

General routing responsibilities:

- sql_agent:
  Aggregate analytics, counts, rates, averages, summaries,
  and other structured-data questions.

- prediction_agent:
  Predict customer churn or classify a support issue,
  depending on the customer request.

- vector_search_agent:
  Retrieve semantically similar customer notes, complaints,
  cancellation reasons, or historical support issues.

- retention_agent:
  Recommend an appropriate retention action. This agent should
  run only when the customer request asks for a recommendation,
  intervention, or retention action.

- final_response_agent:
  Generate the final grounded customer-facing response.
  This agent must always be the final task.

Planning rules:

1. Assign no more than one task to the same agent.
2. Use only agents required by the request.
3. Every dependency must refer to an earlier task.
4. The final_response_agent must always be included.
5. The final_response_agent must always be the final task.
6. The final_response_agent must depend on all specialist agents
   whose results are required for the answer.
7. The retention_agent must depend on the specialist results it
   needs to produce its recommendation.
8. Do not add unnecessary agents.
9. Do not include explanations outside the JSON response.
10. Return valid JSON only.
"""


##### 10. Allowed Schema Values

In [0]:
def get_allowed_literal_values(
    literal_type: Any,
) -> List[str]:
    """
    Return the allowed string values from a Literal type.

    Parameters
    ----------
    literal_type:
        Shared Literal type such as AgentName or RequestType.

    Returns
    -------
    List[str]
        Allowed values defined by the Literal type.
    """

    return [
        value
        for value in get_args(literal_type)
        if isinstance(value, str)
    ]

In [0]:
ALLOWED_AGENT_NAMES = get_allowed_literal_values(
    AgentName
)

ALLOWED_REQUEST_TYPES = get_allowed_literal_values(
    RequestType
)


##### 11. Build the Coordinator Prompt

In [0]:
def build_coordinator_prompt(
    user_request: str,
) -> str:
    """
    Build the prompt used by the LLM for Coordinator Agent planning.

    Parameters
    ----------
    user_request:
        Original customer request.

    Returns
    -------
    str
        Prompt containing instructions, schema constraints,
        and the customer request.
    """

    allowed_agents = json.dumps(
        ALLOWED_AGENT_NAMES,
        indent=2,
    )

    allowed_request_types = json.dumps(
        ALLOWED_REQUEST_TYPES,
        indent=2,
    )

    output_schema = {
        "agent_name": "coordinator_agent",
        "status": "success",
        "message": (
            "A short description confirming that the "
            "execution plan was created."
        ),
        "request_type": (
            "One value from allowed_request_types"
        ),
        "execution_plan": [
            {
                "task_id": "task_1",
                "agent_name": (
                    "One value from allowed_agent_names"
                ),
                "task_description": (
                    "Clear description of the assigned task"
                ),
                "depends_on": [],
            }
        ],
    }

    return f"""
{COORDINATOR_SYSTEM_INSTRUCTIONS}

Allowed agent names:

{allowed_agents}

Allowed request types:

{allowed_request_types}

Required JSON structure:

{json.dumps(output_schema, indent=2)}

Important output requirements:

- Use "coordinator_agent" as agent_name.
- Use "success" as status when the plan is created.
- Use sequential task identifiers:
  task_1, task_2, task_3, and so on.
- The depends_on field must contain agent names, not task IDs.
- Return exactly one JSON object.
- Do not use Markdown code fences.
- Do not include text before or after the JSON object.

Customer request:

{user_request}
""".strip()


##### 12. Extract JSON from the LLM Response

In [0]:
def extract_json_object(
    response_text: str,
) -> Dict[str, Any]:
    """
    Extract one JSON object from an LLM response.

    Parameters
    ----------
    response_text:
        Raw text returned by the LLM.

    Returns
    -------
    Dict[str, Any]
        Parsed JSON object.

    Raises
    ------
    ValueError
        If the response is empty, does not contain a JSON
        object, or contains invalid JSON.
    """

    if not isinstance(response_text, str):
        raise ValueError(
            "Coordinator response must be a string."
        )

    cleaned_response = response_text.strip()

    if not cleaned_response:
        raise ValueError(
            "Coordinator returned an empty response."
        )

    if cleaned_response.startswith("```"):
        lines = cleaned_response.splitlines()

        if lines:
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned_response = "\n".join(lines).strip()

    object_start = cleaned_response.find("{")
    object_end = cleaned_response.rfind("}")

    if object_start == -1 or object_end == -1:
        raise ValueError(
            "Coordinator response does not contain "
            "a JSON object."
        )

    json_text = cleaned_response[
        object_start:object_end + 1
    ]

    try:
        parsed_response = json.loads(json_text)

    except JSONDecodeError as exc:
        raise ValueError(
            "Coordinator returned invalid JSON: "
            f"{exc.msg}"
        ) from exc

    if not isinstance(parsed_response, dict):
        raise ValueError(
            "Coordinator JSON must contain one object."
        )

    return parsed_response


##### 13. Validate the Execution Plan

In [0]:
def validate_execution_plan(
    coordinator_result: CoordinatorResult,
) -> None:
    """
    Validate workflow rules that span multiple tasks.

    Parameters
    ----------
    coordinator_result:
        Pydantic-validated Coordinator result.

    Raises
    ------
    ValueError
        If the execution plan violates workflow rules.
    """

    execution_plan = (
        coordinator_result.execution_plan
    )

    if not execution_plan:
        raise ValueError(
            "Coordinator execution plan cannot be empty."
        )

    task_ids = [
        task.task_id
        for task in execution_plan
    ]

    if len(task_ids) != len(set(task_ids)):
        raise ValueError(
            "Coordinator execution plan contains "
            "duplicate task IDs."
        )

    assigned_agents = [
        task.agent_name
        for task in execution_plan
    ]

    if len(assigned_agents) != len(
        set(assigned_agents)
    ):
        raise ValueError(
            "Coordinator assigned more than one task "
            "to the same agent."
        )

    expected_task_ids = [
        f"task_{index}"
        for index in range(
            1,
            len(execution_plan) + 1,
        )
    ]

    if task_ids != expected_task_ids:
        raise ValueError(
            "Coordinator task IDs must be sequential: "
            "task_1, task_2, task_3, and so on."
        )

    final_task = execution_plan[-1]

    if (
        final_task.agent_name
        != "final_response_agent"
    ):
        raise ValueError(
            "final_response_agent must be the final task."
        )

    if assigned_agents.count(
        "final_response_agent"
    ) != 1:
        raise ValueError(
            "The execution plan must contain exactly one "
            "final_response_agent task."
        )

    previously_assigned_agents = set()

    for task in execution_plan:
        duplicate_dependencies = (
            len(task.depends_on)
            != len(set(task.depends_on))
        )

        if duplicate_dependencies:
            raise ValueError(
                f"Task '{task.task_id}' contains duplicate "
                "dependencies."
            )

        for dependency in task.depends_on:
            if dependency not in previously_assigned_agents:
                raise ValueError(
                    f"Task '{task.task_id}' depends on "
                    f"'{dependency}', but that agent does "
                    "not appear earlier in the plan."
                )

        previously_assigned_agents.add(
            task.agent_name
        )


##### 14. Execute Coordinator Logic

In [0]:
def execute_coordinator_agent(
    user_request: str,
    llm_invoke: LLMInvokeFunction,
) -> CoordinatorResult:
    """
    Analyze a customer request and create a validated plan.

    Parameters
    ----------
    user_request:
        Original customer request.

    llm_invoke:
        Function that accepts a prompt and returns the
        model response as text.

    Returns
    -------
    CoordinatorResult
        Validated Coordinator result.

    Raises
    ------
    ValueError
        If the request, model response, or execution plan
        is invalid.

    ValidationError
        If the response does not match CoordinatorResult.
    """

    cleaned_request = user_request.strip()

    if not cleaned_request:
        raise ValueError(
            "Coordinator requires a non-empty "
            "customer request."
        )

    coordinator_prompt = build_coordinator_prompt(
        user_request=cleaned_request,
    )

    raw_response = llm_invoke(
        coordinator_prompt
    )

    response_payload = extract_json_object(
        response_text=raw_response
    )

    coordinator_result = (
        CoordinatorResult.model_validate(
            response_payload
        )
    )

    if (
        coordinator_result.agent_name
        != "coordinator_agent"
    ):
        raise ValueError(
            "Coordinator result must use "
            "'coordinator_agent' as agent_name."
        )

    if coordinator_result.status != "success":
        raise ValueError(
            "Coordinator result must have success status "
            "when an execution plan is returned."
        )

    validate_execution_plan(
        coordinator_result=coordinator_result
    )

    return coordinator_result


##### 15. Shared-State Runner

In [0]:
def run_coordinator_agent(
    state: MultiAgentState,
    llm_invoke: LLMInvokeFunction,
) -> MultiAgentState:
    """
    Run the Coordinator Agent within the shared workflow.

    The function:

    1. Reads the customer request.
    2. Invokes the Coordinator execution logic.
    3. Stores the validated result.
    4. Records execution history.
    5. Records errors when execution fails.
    6. Returns the updated shared state.

    Parameters
    ----------
    state:
        Current multi-agent shared state.

    llm_invoke:
        Function used to invoke the Coordinator LLM.

    Returns
    -------
    MultiAgentState
        Updated shared workflow state.
    """

    try:
        coordinator_result = (
            execute_coordinator_agent(
                user_request=state["user_request"],
                llm_invoke=llm_invoke,
            )
        )

        state["coordinator_result"] = (
            coordinator_result
        )

        record_agent_execution(
            state=state,
            agent_name="coordinator_agent",
            status="success",
            message=coordinator_result.message,
        )

    except (
        ValueError,
        ValidationError,
        TypeError,
        KeyError,
    ) as exc:
        state["coordinator_result"] = None

        error_message = str(exc)

        record_agent_execution(
            state=state,
            agent_name="coordinator_agent",
            status="failed",
            message=(
                "Coordinator Agent failed to create "
                "a valid execution plan."
            ),
        )

        record_agent_error(
            state=state,
            agent_name="coordinator_agent",
            error_code="COORDINATOR_ERROR",
            error_message=error_message,
        )

    except Exception as exc:
        state["coordinator_result"] = None

        error_message = (
            "Unexpected Coordinator Agent error: "
            f"{exc}"
        )

        record_agent_execution(
            state=state,
            agent_name="coordinator_agent",
            status="failed",
            message=(
                "Coordinator Agent encountered an "
                "unexpected error."
            ),
        )

        record_agent_error(
            state=state,
            agent_name="coordinator_agent",
            error_code=(
                "COORDINATOR_UNEXPECTED_ERROR"
            ),
            error_message=error_message,
        )

    return state


##### 16. Independent Test Function

LLM text → JSON parsing → Python dictionary → Pydantic validation → structured agent result → shared-state error handling

In [0]:
def test_coordinator_agent() -> None:
    """
    Run deterministic unit-style tests for the
    Coordinator Agent.

    The test uses mock LLM responses so it does not
    require a live model endpoint.
    """

    allowed_request_type = (
        ALLOWED_REQUEST_TYPES[0]
    )

    valid_response = {
        "agent_name": "coordinator_agent",
        "status": "success",
        "message": (
            "Execution plan created successfully."
        ),
        "request_type": allowed_request_type,
        "execution_plan": [
            {
                "task_id": "task_1",
                "agent_name": (
                    "final_response_agent"
                ),
                "task_description": (
                    "Generate the final grounded response."
                ),
                "depends_on": [],
            }
        ],
    }

    def mock_valid_llm(
        prompt: str,
    ) -> str:
        assert isinstance(prompt, str)
        assert prompt.strip()

        return json.dumps(valid_response)

    test_state = create_initial_state(
        "Validate the Coordinator workflow."
    )

    updated_state = run_coordinator_agent(
        state=test_state,
        llm_invoke=mock_valid_llm,
    )

    assert (
        updated_state["coordinator_result"]
        is not None
    )

    assert (
        updated_state[
            "coordinator_result"
        ].agent_name
        == "coordinator_agent"
    )

    assert (
        updated_state[
            "coordinator_result"
        ].execution_plan[-1].agent_name
        == "final_response_agent"
    )

    assert len(
        updated_state["execution_history"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][0].status
        == "success"
    )

    assert updated_state["errors"] == []

    def mock_invalid_llm(
        prompt: str,
    ) -> str:
        return "This is not valid JSON."

    failure_state = create_initial_state(
        "Validate Coordinator error handling."
    )

    failed_state = run_coordinator_agent(
        state=failure_state,
        llm_invoke=mock_invalid_llm,
    )

    assert (
        failed_state["coordinator_result"]
        is None
    )

    assert len(
        failed_state["execution_history"]
    ) == 1

    assert (
        failed_state[
            "execution_history"
        ][0].status
        == "failed"
    )

    assert len(
        failed_state["errors"]
    ) == 1

    assert (
        failed_state["errors"][0].error_code
        == "COORDINATOR_ERROR"
    )

    print(
        "All Coordinator Agent tests passed."
    )


##### 17. Key Learnings

- The Coordinator Agent plans work but does not execute specialist tools.

- Planning and execution are separated into different components.

- The LLM returns a structured execution plan rather than a customer-facing answer.

- CoordinatorResult validates the structure of the Coordinator output.

- Additional workflow validation checks relationships across tasks.

- Task dependencies ensure that agents execute only after their required inputs are available.

- The Final Response Agent must always be the final task.

- Dependency injection separates Coordinator logic from a particular LLM endpoint.

- Mock LLM functions make the Coordinator independently testable.

- The shared-state runner separates core planning logic from workflow-state management.

- Execution history and error records improve traceability and debugging.


##### 18. Conclusion

- In this notebook, we implemented the Coordinator Agent responsible for planning the multi-agent workflow.

- The Coordinator analyzes the customer request, identifies the request type, selects the required specialist agents, defines their execution order, and creates task dependencies. Its output is parsed from JSON, validated through CoordinatorResult, checked against workflow-level rules, and stored in shared state.


##### 19. Next Notebook

The next notebook is: 04_sql_agent

The SQL Agent will:

- Read the Coordinator execution plan.

- Determine whether it has an assigned task.

- Skip execution when no SQL task exists.

- Call the existing SQL Analytics Tool when required.

- Validate the result using SQLAgentResult.

- Store its output in agent_results.

- Record execution history.

- Record errors when execution fails.